# Getting Started with MPSFast.jl

This notebook takes you from zero to a trained **Matrix Product State (MPS) Born machine** in five steps:

1. Install / activate the package
2. Generate or load some paths
3. Encode the paths
4. Train the MPS
5. Sample new paths and check the fit

No prior knowledge of quantum mechanics is required — think of an MPS as a very expressive, tractable **joint probability model** over discrete sequences.

## 1. Setup

In [2]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # MPSFast.jl project root
Pkg.resolve()      # writes/updates Manifest.toml (handles stdlib deps correctly)
Pkg.instantiate()  # downloads/installs any missing registered packages

  Activating project at `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl`
     Project No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Project.toml`
    Manifest No packages added to or removed from `~/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/Manifest.toml`


In [3]:
using MPSFast
using MPSFast.Encoders
using Random, LinearAlgebra, Statistics, Printf

println("Threads: ", Threads.nthreads())

Threads: 4


## 2. Generate synthetic paths

We use a simple **Geometric Brownian Motion** (GBM) for illustration.
Replace with real data or a Heston simulation from `experiments/paper_reproduction.ipynb`.

In [4]:
rng = MersenneTwister(42)

N = 5_000    # number of paths
M = 20       # timesteps per path
dt = 1/252
μ  = 0.05
σ  = 0.20
S0 = 100.0

# GBM: S[t+1] = S[t] * exp((μ - σ²/2)dt + σ√dt Z)
Z = randn(rng, N, M)
log_returns = (μ - 0.5 * σ^2) * dt .+ σ * sqrt(dt) .* Z
log_prices  = cumsum(log_returns, dims = 2)
paths       = S0 .* exp.(log_prices)   # (N, M)

println("paths: ", size(paths), "  min=", round(minimum(paths), digits=2),
        "  max=", round(maximum(paths), digits=2))

paths: (5000, 20)  min=81.42  max=124.03


## 3. Encode paths

We discretise each continuous price level into one of `d = 2^m` buckets.
`BasisEncoder` uses one MPS site per timestep with `d`-dimensional physical indices.

In [5]:
m   = 4                        # d = 2^4 = 16 buckets
enc = BasisEncoder(m)
fit_grid!(enc, paths)          # calibrate Smin/Smax to training data

xi = encode_paths(enc, paths)  # (N, M) integer matrix, values in 1:16

encoder_summary(enc, M, 16)    # print a quick summary
println("\nxi shape: ", size(xi), "  unique values: ", sort(unique(xi)))

┌ Info: Encoder
│   encoder = BasisEncoder
│   M = 20
│   chain_length = 20
│   site_dim = 16
│   D_max = 16
│   params = 74240
└ @ MPSFast.Encoders /Users/bi006881/dev/Notes on Time Series Generation for Options Pricing/repos/MPSFast.jl/src/Encoders.jl:414



xi shape: (5000, 20)  unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


## 4. Initialise and train the MPS

The MPS is trained by minimising the **Negative Log-Likelihood** (NLL) via a
DMRG-style sweep with Adam updates at each bond. Each epoch = one forward sweep + one backward sweep.

In [6]:
D_max    = 20      # maximum bond dimension
n_epochs = 30      # 10 was too few; 30 shows clear convergence
η        = 3e-3    # Adam learning rate (5e-4 was too small to see progress in 10 epochs)
ε_cut    = 1e-5    # SVD truncation floor

mps = init_mps(chain_length(enc, M), site_dim(enc), D_max; rng = MersenneTwister(1))

nll_hist = train_mps!(
    mps, xi, n_epochs, η, D_max, ε_cut;
    verbose      = true,
    nll_samples  = 500,
)

train_mps!: Ml=20, Nd=5000, d=16, D_max=20, epochs=30, one-hot
— Epoch 1/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 1/30 | NLL ≈ 26.8372 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 1.021 s
— Epoch 2/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 2/30 | NLL ≈ 20.6858 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 0.49 s
— Epoch 3/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → canonicalize + rebuild envs + backward sweep …
  ↳ backward done → canonicalize + NLL estimate …
Epoch 3/30 | NLL ≈ 19.8408 | bonds=[16,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,16] | 0.574 s
— Epoch 4/30 —
  ↳ norm envs ready → forward sweep (19 bonds) …
  ↳ forward done → can

30-element Vector{Float64}:
 26.837162569331472
 20.685830750901825
 19.84075265733347
 19.643033870696573
 19.291388205576112
 19.27106260745086
 19.0344034630252
 19.08696137147319
 19.146493020518374
 18.6696935839801
  ⋮
 17.60940055277481
 17.426211998387384
 17.602917224329932
 17.59100998555237
 17.343717502642434
 17.27056669192044
 17.56761803495593
 17.354162197818805
 17.43583041098532

In [7]:
# NLL learning curve
println("NLL history: ", round.(nll_hist, digits = 4))
println("Final NLL  : ", round(nll_hist[end], digits = 4))

NLL history: [26.8372, 20.6858, 19.8408, 19.643, 19.2914, 19.2711, 19.0344, 19.087, 19.1465, 18.6697, 18.5328, 18.5477, 18.134, 18.2365, 18.1325, 18.0878, 17.8647, 17.8024, 17.8135, 17.7546, 17.6241, 17.6094, 17.4262, 17.6029, 17.591, 17.3437, 17.2706, 17.5676, 17.3542, 17.4358]
Final NLL  : 17.4358


## 5. Sample and validate

Draw new paths by sequential conditional sampling and compare marginal statistics
to the training data.

In [8]:
n_samp = 2_000
sampled_paths, sampled_xi = sample_paths(enc, mps, n_samp; seed = 99)

println("Sampled paths shape: ", size(sampled_paths))

# Compare per-timestep mean and std
println("\nTimestep  Train mean  Samp mean  Train std  Samp std")
for t in [1, 5, 10, 15, 20]
    @printf("  t=%-3d    %8.3f   %8.3f   %8.3f  %8.3f\n", t, mean(paths[:,t]), mean(sampled_paths[:,t]), std(paths[:,t]), std(sampled_paths[:,t]))
end

Sampled paths shape: (2000, 20)

Timestep  Train mean  Samp mean  Train std  Samp std
  t=1       100.004     98.710      1.257     3.047
  t=5       100.053     98.566      2.816     3.062
  t=10      100.112     98.766      3.979     4.408
  t=15      100.216     98.919      4.870     5.095
  t=20      100.255     98.933      5.611     5.799


In [9]:
# Bond dimensions after training
bonds = [size(mps[j], 3) for j in 1:length(mps)-1]
println("Bond dimensions: ", bonds)

Bond dimensions: [16, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 16]


In [10]:
# Bipartite von-Neumann entropy profile
Svals, entropies = bipartite_entropies(mps)
println("Bipartite entropies (nats): ", round.(entropies, digits = 3))

Bipartite entropies (nats): [0.742, 1.305, 1.609, 1.466, 1.525, 1.573, 1.437, 1.329, 1.74, 1.765, 1.853, 1.891, 1.925, 1.959, 1.967, 2.027, 2.06, 2.113, 1.902]


## Next steps

- **`notebooks/dmrg_tutorial.ipynb`** — step-by-step derivation of the DMRG algorithm.
- **`notebooks/experiments/paper_reproduction.ipynb`** — full Heston model reproduction and options pricing.
- **`notebooks/experiments/encodings_comparison.ipynb`** — compare BasisEncoder vs BinaryEncoder vs TrigEncoder.
- **`notebooks/experiments/classification.ipynb`** — supervised Born-machine classification.